In [73]:
import os
import wikipediaapi
import time
import json
import re

In [125]:
target_categories = {
    "anime":"各年のテレビアニメ",
    "manga":"漫画作品 (五十音別)",
    "light_novel":"ライトノベル",
    "game":"日本で開発されたコンピュータゲーム"
    }
user_agent = user_agent='AnimeRAGBot/1.0 (mrluettgen@gmail.com)'
folder = 'wiki_data_json'

In [98]:
if not os.path.exists(f"/{folder}"):
    os.mkdir(f"/{folder}")
for cat in ('anime', 'manga', 'light_novel', 'game'):
    if not os.path.exists(f'/{folder}/{cat}'):
        os.mkdir(f"/{folder}/{cat}")

In [99]:
def extract_english_title_from_content(text):
    """
    Extracts the English title from the Japanese Wikipedia lead sentence.
    Target format: 英語: Is the order a rabbit?
    """
    if not text:
        return None

    # Regex breakdown:
    # 英語:\s* -> Matches '英語:' followed by any optional spaces
    # ([a-zA-Z0-9\s\-\!\?\'\.\,\:\;\&\"]+) -> Captures English letters, numbers, spaces, and punctuation
    # (?=[）\),]) -> Positive lookahead to stop capturing when hitting a closing parenthesis or comma
    pattern = r"英語:\s*([a-zA-Z0-9\s\-\!\?\'\.\,\:\;\&\"]+)(?=[）\),])"

    match = re.search(pattern, text)

    if match:
        # Extract the captured group and strip trailing/leading spaces
        return match.group(1).strip()

    # Return None if the article doesn't have an English title marker
    return None

# --- Quick Test ---
sample_text = '『ご注文はうさぎですか？』（ごちゅうもんはうさぎですか？、英語: Is the order a rabbit?）は、Koiによる同名の4コマ漫画を原作としたテレビアニメおよびOVA。'
sample_text_nonexistent = '『ご注文はうさぎですか？』（ごちゅうもんはうさぎですか？）は、Koiによる同名の4コマ漫画を原作としたテレビアニメおよびOVA。'

print(f"Extracted Title: '{extract_english_title_from_content(sample_text)}'")
print(f"Extracted Title: '{extract_english_title_from_content(sample_text_nonexistent)}'")

def extract_english_title(article):
    try:
        if article and hasattr(article, 'langlinks') and 'en' in article.langlinks:
            en_page = article.langlinks['en']
            if en_page and en_page.title:
                return en_page.title.strip()
    except Exception:
        pass
    return extract_english_title_from_content(article.text)


# Output: Extracted Title: 'Is the order a rabbit?'

Extracted Title: 'Is the order a rabbit?'
Extracted Title: 'None'


In [77]:
def extract_first_year(text):
    """
    Extracts the very first year instance from Japanese Wikipedia.
    Guarantees returning a clean, half-width integer (e.g., 2014).
    """
    if not text:
        return None

    # Step 1: Look strictly for 2 to 4 digits followed immediately by '年'
    # [0-90-9] captures both standard English numbers and Japanese full-width numbers
    pattern = r"([0-9０-９]{2,4})年"

    match = re.search(pattern, text)

    if match:
        year_str = match.group(1)
        # Step 2: Translate full-width '２０１４' -> half-width '2014'
        full_width_digits = '０１２３４５６７８９'
        half_width_digits = '0123456789'
        translation_table = str.maketrans(full_width_digits, half_width_digits)

        normalized_str = year_str.translate(translation_table)
        return int(normalized_str)

    return None

# --- Re-testing the Framework ---
tests = [
    "『ご注文はうさぎですか？』は、2011年より芳文社で連載...",
    "本作は２０１４年にテレビアニメ化され、大ヒットを記録した。",
    "アニメ第1期は2014年（平成26年）4月から6月まで放送された。",
    "アニメ",
    "『アニメンタリー 決断』（アニメンタリー けつだん）は、太平洋戦争を題材にした竜の子プロダクション制作のテレビアニメである。1971年4月3日から同年9月25日までの間、毎週土曜日19時30分 - 20時に日本テレビ系で全26回放映された[1]。"
]

for i, test in enumerate(tests, 1):
    print(f"Test {i} Result: {extract_first_year(test)}")

Test 1 Result: 2011
Test 2 Result: 2014
Test 3 Result: 2014
Test 4 Result: None
Test 5 Result: 1971


In [118]:
##ai wrote this too.
def extract_top_characters(article_page, max_chars=5):
    """
    Extracts the top character names from a Wikipedia page object
    by targeting the '登場人物' or 'キャスト' section headers.
    """
    characters = []

    # Target sections common in Japanese media wikis
    target_sections = ["登場人物", "キャスト", "キャラクター"]
    char_section = None

    # 1. Look for the matching section object
    for section in article_page.sections:
        if section.title in target_sections:
            char_section = section
            break

    if char_section:
        if char_section.sections:
            char_sections = char_section.sections
        else:
            char_sections  = [char_section]
    else:
        char_sections = []

    for char_section_2 in char_sections:
        # 2. Extract lines from that section
        # Character entries on wiki pages almost always start with '・' or ' ' line items
        lines = char_section_2.text.split('\n')
        for line in lines:
            line = line.strip()
            if not line:
                continue

            # Wikipedia character headers usually look like: "主人公名前（しゅじんこう） - 声: 声優"
            # We look for the main name before the opening bracket or description split
            # Split by common entry punctuation marks: '（', '(', or ' -'
            name_match = re.split(r'[（\(\-：:]', line)[0]

            # Clean up residual list symbols
            name_match = name_match.replace('・', '').strip()

            # Basic validation: ensure it's a realistic name length (2 to 10 chars)
            if 2 <= len(name_match) <= 10 and name_match not in characters:
                characters.append(name_match)

            if len(characters) >= max_chars:
                break
        if len(characters) >= max_chars:
            break

    return characters

In [79]:
import re

def extract_author(text):
    """
    Extracts the author/creator name from the Wikipedia lead sentence.
    Targets standard Japanese phrasing structures (〜による, 〜原作の).
    """
    if not text:
        return None

    # Pattern 1: targeting "[Author Name]による" (written by [Author])
    # Pattern 2: targeting "[Author Name]の漫画を原作とした" (based on the manga by [Author])
    patterns = [
        r"（[^）]+）\s*は、([^）\s、]+)による",
        r"は、([^）\s、]+)による",
        r"は、([^）\s、]+)の(?:漫画|ライトノベル|小説)を原作"
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            author_name = match.group(1).strip()

            # Clean up trailing punctuation if any slipped through
            author_name = re.sub(r"[は、。』『]（", "", author_name)

            # Simple length validation to filter out noise (Japanese names are usually 2-6 chars)
            if 2 <= len(author_name) <= 8 and "作品" not in author_name:
                return author_name

    return None

# --- Quick Test ---
sample_text = '『ご注文はうさぎですか？』は、Koiによる同名の4コマ漫画を原作としたテレビアニメ。'
print(f"Extracted Author: {extract_author(sample_text)}")
# Output: Extracted Author: Koi



Extracted Author: Koi


In [120]:
all_countries_japanese = [
    # --- アジア (Asia - 48ヶ国) ---
    "韓国", "中国", "北朝鮮", "モンゴル",
    "シンガポール", "マレーシア", "タイ", "インドネシア", "フィリピン",
    "ベトナム", "カンボジア", "ラオス", "ミャンマー", "東ティモール",
    "インド", "パキスタン", "バングラデシュ", "スリランカ", "ネパール",
    "ブータン", "モルディブ", "ウズベキスタン", "カザフスタン", "キルギス",
    "タジキスタン", "トルクメニスタン", "アフガニスタン", "イラン", "イラク",
    "サウジアラビア", "アラブ首長国連邦", "カタール", "クウェート", "バーレーン",
    "オマーン", "イエメン", "シリア", "ヨルダン", "レバノン",
    "イスラエル", "パレスチナ", "トルコ", "キプロス", "ジョージア",
    "アルメニア", "アゼルバイジャン", "台湾",

    # --- ヨーロッパ (Europe - 44ヶ国) ---
    "イギリス", "フランス", "ドイツ", "イタリア", "スペイン",
    "ポルトガル", "オランダ", "ベルギー", "ルクセンブルク", "アイルランド",
    "スイス", "オーストリア", "スウェーデン", "ノルウェー", "フィンランド",
    "デンマーク", "アイスランド", "ポーランド", "チェコ", "スロバキア",
    "ハンガリー", "ルーマニア", "ブルガリア", "ギリシャ", "セルビア",
    "クロアチア", "スロベニア", "ボスニア・ヘルツェゴビナ", "モンテネグロ", "北マケドニア",
    "アルバニア", "ウクライナ", "ベラルーシ", "モルドバ", "ロシア",
    "エストニア", "ラトビア", "リトアニア", "マルタ", "サンマリノ",
    "バチカン市国", "アンドラ", "モナコ", "リヒテンシュタイン",

    # --- アフリカ (Africa - 54ヶ国) ---
    "エジプト", "南アフリカ", "ナイジェリア", "ケニア", "モロッコ",
    "アルジェリア", "チュニジア", "リビア", "スーダン", "南スーダン",
    "エチオピア", "ソマリア", "エリトリア", "ジブチ", "ウガンダ",
    "タンザニア", "ルワンダ", "ブルンジ", "セーシェル", "マダガスカル",
    "モーリシャス", "コモロ", "マラウイ", "ザンビア", "ジンバブエ",
    "ボツワナ", "ナミビア", "レソト", "エスワティニ", "モザンビーク",
    "アンゴラ", "コンゴ民主共和国", "コンゴ共和国", "ガボン", "赤道ギニア",
    "カメルーン", "中央アフリカ", "チャド", "ニジェール", "マリ",
    "モーリタニア", "セネガル", "ガンビア", "ギニア・ビサウ", "ギニア",
    "シエラレオネ", "リベリア", "コートジボワール", "ガーナ", "トーゴ",
    "ベナン", "ブルキナファソ", "カーボベルデ", "サントメ・プリンシペ",

    # --- 北中米 (North & Central America - 23ヶ国) ---
    "アメリカ", "カナダ", "メキシコ", "グアテマラ", "ベリーズ",
    "エルサルバドル", "ホンジュラス", "ニカラグア", "コスタリカ", "パナマ",
    "キューバ", "ジャマイカ", "バハマ", "ハイチ", "ドミニカ",
    "アンティグア・バーブーダ", "セントクリストファー・ネイビス", "ドミニカ", "セントルシア", "セントビンセント・グレナディーン",
    "バルバドス", "グレナダ", "トリニダード・トバゴ",

    # --- 南米 (South America - 12ヶ国) ---
    "ブラジル", "アルゼンチン", "コロンビア", "ペルー", "チリ",
    "ベネズエラ", "エクアドル", "ボリビア", "パラグアイ", "ウルグアイ",
    "ガイアナ", "スリナム",

    # --- オセアニア (Oceania - 14ヶ国) ---
    "オーストラリア", "ニュージーランド", "パプアニューギニア", "フィジー", "ソロモン諸島",
    "バヌアツ", "サモア", "トンガ", "ツバル", "キリバス",
    "ナウル", "ミクロネシア連邦", "マーシャル諸島", "パラオ"
]

def is_foreign_work(text, author_name):
    """
    Checks if a Wikipedia page belongs to a foreign/imported work.
    """
    if not text:
        return True
    intro = text[:300]
    if "原題" in intro:
        return True
    if any(marker in intro for marker in all_countries_japanese) and ("日本" not in intro.replace("日本で", "").replace("日本語","")):
        return True

    # 2. Check if the author name contains a foreign katakana middle dot
    # e.g., 'ウィリアム・ハンナ' -> True
    if author_name and "・" in author_name:
        return True
    return False

In [126]:
# 1. Initialize the Wikipedia object with a descriptive user agent
# Replace the email below with your own as per Wikipedia's policy
wiki_ja = wikipediaapi.Wikipedia(
    user_agent=user_agent,
    language='ja'
)

# Keep track of visited categories to avoid infinite loops/redundancy
visited_categories = set()


# 2. Define a function to get all pages in a category
def get_category_members(category_name, level=0, max_level=3):
    if "Category:" not in category_name:
        category_name = f"Category:{category_name}"
    if category_name in visited_categories or level > max_level:
        return []
    visited_categories.add(category_name)
    time.sleep(1)
    cat = wiki_ja.page(category_name)
    pages = []

    if not cat.exists():
        return pages

    for member in cat.categorymembers.values():
        # ns=0 is the namespace for standard articles
        if member.ns == wikipediaapi.Namespace.MAIN:
            if "一覧" not in member.title and "年代" not in member.title:
                pages.append(member)
        # Optional: recursively fetch subcategories up to a certain depth
        elif member.ns == wikipediaapi.Namespace.CATEGORY and level < max_level:
            pages.extend(get_category_members(member.title, level + 1, max_level))

    return pages

def get_test_articles(titles):
    return [wiki_ja.page(t) for t in titles]
# 3. Targeted categories (Japanese titles for better accuracy)


for en_cat, jp_cat in target_categories.items():
    print(f"Fetching articles for: {en_cat}...")
    articles = get_category_members(jp_cat)
    # articles = get_test_articles(["葬送のフリーレン", "ひぐらしのなく頃に", "お兄ちゃんはおしまい!"])
    unique_articles = {a.title: a for a in articles}
    article_list = list(unique_articles.keys())
    folder_path = f"/{folder}/{en_cat}"
    print(f"Downloading {len(article_list)} articles to {folder_path}...")
    for title in article_list:
        # Sanitize filename (remove characters like / or :)
        article = unique_articles[title]
        safe_title = title.replace("/", "_").replace(":", "_")
        file_path = f"{folder_path}/{safe_title}.json"
        # SKIP if already exists (Resume-ability)
        # if not os.path.exists(file_path):
        #   continue
        # SKIP if does not already exist (Re Download)
        if not os.path.exists(file_path):
            continue
        try:
            time.sleep(1)
            # article.text triggers the API fetch for the full content
            page_content = article.text
            author = extract_author(page_content)
            if is_foreign_work(page_content, author):
                continue
            jsn = {"page_content": page_content,
                   "metadata":{
                       'title_ja': article.title,
                       'title_en': extract_english_title(article),
                       'year': extract_first_year(page_content),
                       'characters': extract_top_characters(article),
                       'mediatype': en_cat,
                       'author': extract_author(page_content),
                       'url': article.fullurl,
                       'source': safe_title
                       }
                   }

            with open(file_path, "w", encoding="utf-8") as file:
                file.write(json.dumps(jsn,ensure_ascii=False, indent=2))
            print(f"Saved: {title}")
        except Exception as e:
            print(f"Error saving {title}: {e}")

Fetching articles for: anime...
Saved: ウッディー・ウッドペッカー・ショー
Saved: 漫画ニュース
Saved: もぐらのアバンチュール
Saved: クラッチ・カーゴ
Saved: テレビ坊やの冒険
Saved: ロッキーとブルウィンクルの大冒険
Saved: 新しい動画 3つのはなし
Saved: インスタントヒストリー
Saved: おとぎマンガカレンダー
Saved: リッピーとハーディー
Saved: わんわん保安官
Saved: エイトマン
Saved: 狼少年ケン
Saved: 進めシスコン
Saved: 仙人部落
Saved: 鉄人28号 (テレビアニメ第1作)
Saved: 鉄腕アトム (アニメ第1作)
Saved: ピーコック劇場 (テレビ番組)
Saved: 0戦はやと
Saved: ビッグX
Saved: おかしなおかしな トムとジェリー 大行進
Saved: 新トムとジェリー
Saved: トム・アンド・ジェリー・イン・フィスト・オブ・ファーリー
Saved: とむとじぇりーごっこ
Saved: トムとジェリー大行進
Saved: トムとジェリー 火星へ行く
Saved: トムとジェリーのくるみ割り人形
Saved: トムとジェリー シャーロック・ホームズ
Saved: トムとジェリー魔法の指輪
Saved: ジェリー (トムとジェリー)
Saved: タフィー (トムとジェリー)
Saved: トム (トムとジェリー)
Saved: ブッチ (トムとジェリー)
Saved: 宇宙エース
Saved: 宇宙少年ソラン
Saved: 宇宙パトロールホッパ
Saved: ウッディー・ウッドペッカー
Saved: オバケのQ太郎 (アニメ)
Saved: 怪盗プライド
Saved: キャプテン・ファドム
Saved: クルーゾー警部 (アニメ)
Saved: ジャングル大帝
Saved: 新宝島 (テレビアニメ)
Saved: スーパージェッター
Saved: スヌーピーのメリークリスマス
Saved: 戦え!オスパー
Saved: ドルフィン王子
Saved: ハッスルパンチ
Saved: 遊星少年パピイ
Saved: 遊星ぼうやドド
Saved: W3
Saved: おそ松くん
Saved: 海賊王子

  ## print the number of articles, without pulling all and saving.

In [5]:
wiki_ja = wikipediaapi.Wikipedia(
    user_agent=user_agent,
    language='ja'
)

# Keep track of visited categories to avoid infinite loops/redundancy
visited_categories = set()


# 2. Define a function to get all pages in a category
def get_category_members(category_name, level=0, max_level=3):
    if "Category:" not in category_name:
        category_name = f"Category:{category_name}"
    if category_name in visited_categories or level > max_level:
        return []
    visited_categories.add(category_name)
    time.sleep(1)
    cat = wiki_ja.page(category_name)
    pages = []

    if not cat.exists():
        return pages

    for member in cat.categorymembers.values():
        # ns=0 is the namespace for standard articles
        if member.ns == wikipediaapi.Namespace.MAIN:
            if "一覧" not in member.title and "年代" not in member.title:
                pages.append(member)
        # Optional: recursively fetch subcategories up to a certain depth
        elif member.ns == wikipediaapi.Namespace.CATEGORY and level < max_level:
            pages.extend(get_category_members(member.title, level + 1, max_level))

    return pages

for en_cat, jp_cat in target_categories.items():
    print(f"Fetching articles for: {en_cat}...")
    articles = get_category_members(jp_cat)
    print(en_cat, len(articles))

Fetching articles for: anime...
anime 7674
Fetching articles for: manga...
manga 19892
Fetching articles for: light_novel...
light_novel 2556
Fetching articles for: game...
game 16186
